# 01 — Data Preparation

This notebook rebuilds the La Liga 2010–2020 data preparation pipeline.

It performs:

1. raw CSV ingestion
2. match-level cleaning
3. clean-data validation
4. participation table creation
5. standings calculation
6. standings validation
7. score-only team season stats creation
8. team season stats validation
9. processed dataset export
10. final validation summary

**Project scope:** this version focuses on score-based analysis and team performance.  
Shots, cards and corners are intentionally excluded from the core processed datasets.

## 0. Notebook setup

We add the project root to `sys.path` so the notebook can import from `src/`, even though the notebook is inside the `notebooks/` folder.

`autoreload` helps Jupyter pick up changes made in `.py` files without restarting the kernel every time.

In [1]:
from pathlib import Path
import sys

# Automatically reload imported modules when .py files change
%load_ext autoreload
%autoreload 2

# Add project root to Python path so we can import from src/
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root configured successfully.")

Project root configured successfully.


## 1. Imports

In [2]:
import pandas as pd

from src.config import PROCESSED_DATA_DIR
from src.load_data import get_raw_files, load_all_matches
from src.clean_data import clean_all_matches
from src.features import (
    create_participations_table,
    create_team_participation_counts,
    create_team_season_stats,
)
from src.standings import create_team_match_rows, create_standings_table
from src.validation import (
    run_all_validation_checks,
    run_standings_validation_checks,
    run_team_season_stats_validation_checks,
)

## 2. Helper functions

These helpers keep the notebook output readable.

In [3]:
def validation_passed(validation_report: dict[str, pd.DataFrame]) -> bool:
    """
    Return True if all validation checks with an is_valid column passed.
    """

    checks = []

    for check_df in validation_report.values():
        if "is_valid" in check_df.columns:
            checks.append(check_df["is_valid"].all())

    return all(checks)


def display_validation_report(
    validation_report: dict[str, pd.DataFrame],
    show_only_failures: bool = True
) -> None:
    """
    Display validation report tables.

    If show_only_failures=True, only failed rows are displayed.
    This keeps the notebook readable when all checks pass.
    """

    for check_name, check_df in validation_report.items():
        print("=" * 80)
        print(check_name)
        print("=" * 80)

        if "is_valid" in check_df.columns and show_only_failures:
            failed_rows = check_df.loc[~check_df["is_valid"]]

            if failed_rows.empty:
                print("Passed")
            else:
                display(failed_rows)
        else:
            display(check_df)

## 3. Load raw season files

Each raw CSV file represents one La Liga season.  
The loading layer adds:

- `season`
- `source_file`

Expected result: 10 raw files and 3,800 total matches.

In [4]:
raw_files = get_raw_files()

print(f"Raw files found: {len(raw_files)}")
for file in raw_files:
    print(file.name)

assert len(raw_files) == 10

Raw files found: 10
2010-11_SP1.csv
2011-12_SP1.csv
2012-13_SP1.csv
2013-14_SP1.csv
2014-15_SP1.csv
2015-16_SP1.csv
2016-17_SP1.csv
2017-18_SP1.csv
2018-19_SP1.csv
2019-20_SP1.csv


In [5]:
raw_matches = load_all_matches()

print(f"Raw matches shape: {raw_matches.shape}")
display(raw_matches.head())

assert raw_matches.shape[0] == 3800
assert raw_matches["season"].nunique() == 10
assert raw_matches["source_file"].nunique() == 10

Raw matches shape: (3800, 140)


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,AvgC<2.5,AHCh,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA
0,SP1,28/08/10,Hercules,Ath Bilbao,0,1,A,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SP1,28/08/10,Levante,Sevilla,1,4,A,1,2,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SP1,28/08/10,Malaga,Valencia,1,3,A,1,1,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SP1,29/08/10,Espanol,Getafe,3,1,H,1,0,H,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SP1,29/08/10,La Coruna,Zaragoza,0,0,D,0,0,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
raw_matches["season"].value_counts().sort_index()

season
2010-11    380
2011-12    380
2012-13    380
2013-14    380
2014-15    380
2015-16    380
2016-17    380
2017-18    380
2018-19    380
2019-20    380
Name: count, dtype: int64

## 4. Clean match-level data

The cleaning layer:

- renames raw columns to readable snake_case names
- parses dates using explicit date formats
- keeps score-only core columns
- adds official team names
- validates the full-time result against the score

Expected result: 3,800 clean match rows.

In [7]:
clean_matches = clean_all_matches(raw_matches)

print(f"Clean matches shape: {clean_matches.shape}")
display(clean_matches.head())

Clean matches shape: (3800, 12)


,season,date,home_team,away_team,home_goals,away_goals,result,source_file,home_team_official,away_team_official,expected_result,result_is_valid
0,2010-11,2010-08-28,Hercules,Ath Bilbao,0,1,A,2010-11_SP1.csv,Hércules CF,Athletic Bilbao,A,True
1,2010-11,2010-08-28,Levante,Sevilla,1,4,A,2010-11_SP1.csv,Levante UD,Sevilla FC,A,True
2,2010-11,2010-08-28,Malaga,Valencia,1,3,A,2010-11_SP1.csv,Málaga CF,Valencia CF,A,True
3,2010-11,2010-08-29,Espanol,Getafe,3,1,H,2010-11_SP1.csv,RCD Espanyol,Getafe CF,H,True
4,2010-11,2010-08-29,La Coruna,Zaragoza,0,0,D,2010-11_SP1.csv,Deportivo La Coruña,Real Zaragoza,D,True


In [8]:
# Core cleaning checks
assert clean_matches.shape[0] == 3800
assert clean_matches["date"].isna().sum() == 0
assert clean_matches["result_is_valid"].all()

critical_columns = [
    "season",
    "date",
    "home_team",
    "away_team",
    "home_goals",
    "away_goals",
    "result",
]

assert clean_matches[critical_columns].isna().sum().sum() == 0

In [9]:
# Check missing official team-name mappings
missing_home_names = clean_matches.loc[
    clean_matches["home_team_official"].isna(),
    "home_team"
].unique()

missing_away_names = clean_matches.loc[
    clean_matches["away_team_official"].isna(),
    "away_team"
].unique()

missing_team_names = sorted(
    set(missing_home_names) | set(missing_away_names)
)

missing_team_names

[]

In [10]:
assert missing_team_names == []

In [11]:
# Date range sanity check by season
clean_matches.groupby("season")["date"].agg(["min", "max"])

,min,max
season,,
2010-11,2010-08-28,2011-05-21
2011-12,2011-08-27,2012-05-13
2012-13,2012-08-18,2013-06-01
2013-14,2013-08-17,2014-05-18
2014-15,2014-08-23,2015-05-23
2015-16,2015-08-21,2016-05-15
2016-17,2016-08-19,2017-05-21
2017-18,2017-08-18,2018-05-20
2018-19,2018-08-17,2019-05-19


## 5. Validate clean matches

Validation checks are used as a lightweight data contract.

Expected result: all checks should pass.

In [12]:
clean_matches_validation_report = run_all_validation_checks(clean_matches)

display_validation_report(
    clean_matches_validation_report,
    show_only_failures=True
)

clean_matches_checks_passed = validation_passed(clean_matches_validation_report)

print(f"All clean matches validation checks passed: {clean_matches_checks_passed}")

assert clean_matches_checks_passed

matches_per_season
Passed
teams_per_season
Passed
nulls
Passed
results_are_valid
Passed
score_values
Passed
duplicate_matches
Passed
All clean matches validation checks passed: True


## 6. Create participation datasets

We create two related outputs:

1. `participations.csv`: one row per team per season
2. `team_participation_counts.csv`: one row per team with the number of seasons played

This avoids mixing a detailed participation table with a summary table under the same filename.

In [13]:
participations = create_participations_table(clean_matches)

print(f"Participations shape: {participations.shape}")
display(participations.head())

assert participations.shape[0] == 200
assert participations.duplicated(subset=["season", "team"]).sum() == 0
assert (participations.groupby("season")["team"].nunique() == 20).all()

Participations shape: (200, 2)


,season,team
0,2010-11,Almeria
1,2010-11,Ath Bilbao
2,2010-11,Ath Madrid
3,2010-11,Barcelona
4,2010-11,Espanol


In [14]:
team_participation_counts = create_team_participation_counts(participations)

print(f"Team participation counts shape: {team_participation_counts.shape}")
display(team_participation_counts.head(10))

assert team_participation_counts["team_official"].notna().all()
assert team_participation_counts["team"].nunique() == 33

Team participation counts shape: (33, 3)


,team,team_official,seasons_played
0,Ath Bilbao,Athletic Bilbao,10
1,Ath Madrid,Club Atlético de Madrid,10
2,Barcelona,FC Barcelona,10
3,Espanol,RCD Espanyol,10
4,Real Madrid,Real Madrid CF,10
5,Sevilla,Sevilla FC,10
6,Sociedad,Real Sociedad,10
7,Valencia,Valencia CF,10
8,Getafe,Getafe CF,9
9,Levante,Levante UD,9


## 7. Create standings table

The standings table is calculated from match results.

Ranking assumption:

- points
- goal difference
- goals for
- team name

Official La Liga head-to-head tie-breakers are not implemented in this version.

In [15]:
team_match_rows = create_team_match_rows(clean_matches)

print(f"Team-match rows shape: {team_match_rows.shape}")
display(team_match_rows.head())

assert team_match_rows.shape[0] == clean_matches.shape[0] * 2

Team-match rows shape: (7600, 11)


,season,date,team,opponent,goals_for,goals_against,venue,win,draw,loss,points
0,2010-11,2010-08-28,Hercules,Ath Bilbao,0,1,home,False,False,True,0
1,2010-11,2010-08-28,Levante,Sevilla,1,4,home,False,False,True,0
2,2010-11,2010-08-28,Malaga,Valencia,1,3,home,False,False,True,0
3,2010-11,2010-08-29,Espanol,Getafe,3,1,home,True,False,False,3
4,2010-11,2010-08-29,La Coruna,Zaragoza,0,0,home,False,True,False,1


In [16]:
standings = create_standings_table(clean_matches)

print(f"Standings shape: {standings.shape}")
display(standings.head(20))

assert standings.shape[0] == 200
assert "team_official" in standings.columns
assert standings["team_official"].notna().all()

Standings shape: (200, 12)


,season,position,team,team_official,played,wins,draws,losses,goals_for,goals_against,goal_difference,points
0,2010-11,1,Barcelona,FC Barcelona,38,30,6,2,95,21,74,96
1,2010-11,2,Real Madrid,Real Madrid CF,38,29,5,4,102,33,69,92
2,2010-11,3,Valencia,Valencia CF,38,21,8,9,64,44,20,71
3,2010-11,4,Villarreal,Villarreal CF,38,18,8,12,54,44,10,62
4,2010-11,5,Ath Madrid,Club Atlético de Madrid,38,17,7,14,62,53,9,58
5,2010-11,6,Ath Bilbao,Athletic Bilbao,38,18,4,16,59,55,4,58
6,2010-11,7,Sevilla,Sevilla FC,38,17,7,14,62,61,1,58
7,2010-11,8,Espanol,RCD Espanyol,38,15,4,19,46,55,-9,49
8,2010-11,9,Osasuna,CA Osasuna,38,13,8,17,45,46,-1,47
9,2010-11,10,Sp Gijon,Sporting Gijón,38,11,14,13,35,42,-7,47


## 8. Validate standings

These checks confirm that the standings table reconciles with the match-level data.

In [17]:
standings_validation_report = run_standings_validation_checks(
    matches=clean_matches,
    standings=standings
)

display_validation_report(
    standings_validation_report,
    show_only_failures=True
)

standings_checks_passed = validation_passed(standings_validation_report)

print(f"All standings validation checks passed: {standings_checks_passed}")

assert standings_checks_passed

standings_teams_per_season
Passed
standings_played_matches
Passed
standings_record_totals
Passed
standings_goal_difference
Passed
standings_goals_balance
Passed
standings_points_reconciliation
Passed
standings_official_team_names
Passed
All standings validation checks passed: True


## 9. Create score-only team season stats

This dataset expands the standings with total, home and away performance metrics.

Included metrics:

- wins, draws, losses
- points
- goals for / against
- clean sheets
- failed to score
- home / away splits

Excluded from core scope:

- shots
- shots on target
- cards
- corners

In [18]:
team_season_stats = create_team_season_stats(clean_matches)

print(f"Team season stats shape: {team_season_stats.shape}")
display(team_season_stats.head(20))

assert team_season_stats.shape[0] == 200
assert team_season_stats["team_official"].notna().all()

Team season stats shape: (200, 42)


,season,team,team_official,matches,wins,draws,losses,points,points_per_match,goals_for,...,away_losses,away_points,away_points_per_match,away_goals_for,away_goals_against,away_goal_difference,away_goals_for_per_match,away_goals_against_per_match,away_clean_sheets,away_failed_to_score
0,2010-11,Barcelona,FC Barcelona,38,30,6,2,96,2.53,95,...,1,46,2.42,49,11,38,2.58,0.58,9,0
1,2010-11,Real Madrid,Real Madrid CF,38,29,5,4,92,2.42,102,...,2,43,2.26,41,21,20,2.16,1.11,6,5
2,2010-11,Valencia,Valencia CF,38,21,8,9,71,1.87,64,...,5,36,1.89,30,23,7,1.58,1.21,4,4
3,2010-11,Villarreal,Villarreal CF,38,18,8,12,62,1.63,54,...,9,20,1.05,21,30,-9,1.11,1.58,5,7
4,2010-11,Ath Madrid,Club Atlético de Madrid,38,17,7,14,58,1.53,62,...,8,25,1.32,27,33,-6,1.42,1.74,3,5
5,2010-11,Ath Bilbao,Athletic Bilbao,38,18,4,16,58,1.53,59,...,10,21,1.11,27,35,-8,1.42,1.84,2,2
6,2010-11,Sevilla,Sevilla FC,38,17,7,14,58,1.53,62,...,9,24,1.26,27,34,-7,1.42,1.79,2,7
7,2010-11,Espanol,RCD Espanyol,38,15,4,19,49,1.29,46,...,13,14,0.74,13,33,-20,0.68,1.74,3,12
8,2010-11,Osasuna,CA Osasuna,38,13,8,17,47,1.24,45,...,14,11,0.58,17,32,-15,0.89,1.68,3,11
9,2010-11,Sp Gijon,Sporting Gijón,38,11,14,13,47,1.24,35,...,9,14,0.74,12,26,-14,0.63,1.37,5,11


## 10. Validate team season stats

These checks confirm:

- 20 teams per season
- 38 matches per team
- 19 home and 19 away matches
- home + away totals reconcile with total stats
- team season stats agree with standings

In [19]:
team_season_stats_validation_report = run_team_season_stats_validation_checks(
    team_season_stats=team_season_stats,
    standings=standings
)

display_validation_report(
    team_season_stats_validation_report,
    show_only_failures=True
)

team_season_stats_checks_passed = validation_passed(
    team_season_stats_validation_report
)

print(
    "All team season stats validation checks passed: "
    f"{team_season_stats_checks_passed}"
)

assert team_season_stats_checks_passed

team_season_stats_rows
Passed
team_season_stats_teams_per_season
Passed
team_season_stats_matches
Passed
team_season_stats_record_totals
Passed
team_season_stats_home_away_totals
Passed
team_season_stats_official_names
Passed
team_season_stats_against_standings
Passed
All team season stats validation checks passed: True


## 11. Export processed datasets

The processed datasets are regenerated from the raw files and saved to `data/processed/`.

In [20]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

all_matches_path = PROCESSED_DATA_DIR / "all_matches.csv"
participations_path = PROCESSED_DATA_DIR / "participations.csv"
team_participation_counts_path = PROCESSED_DATA_DIR / "team_participation_counts.csv"
standings_path = PROCESSED_DATA_DIR / "standings.csv"
team_season_stats_path = PROCESSED_DATA_DIR / "team_season_stats.csv"

clean_matches.to_csv(all_matches_path, index=False)
participations.to_csv(participations_path, index=False)
team_participation_counts.to_csv(team_participation_counts_path, index=False)
standings.to_csv(standings_path, index=False)
team_season_stats.to_csv(team_season_stats_path, index=False)

print("Saved processed datasets:")
print(f"- data/processed/{all_matches_path.name}")
print(f"- data/processed/{participations_path.name}")
print(f"- data/processed/{team_participation_counts_path.name}")
print(f"- data/processed/{standings_path.name}")
print(f"- data/processed/{team_season_stats_path.name}")

Saved processed datasets:
- data/processed/all_matches.csv
- data/processed/participations.csv
- data/processed/team_participation_counts.csv
- data/processed/standings.csv
- data/processed/team_season_stats.csv


In [21]:
assert all_matches_path.exists()
assert participations_path.exists()
assert team_participation_counts_path.exists()
assert standings_path.exists()
assert team_season_stats_path.exists()

## 12. Final validation summary

All processed datasets should be reproducible from the raw season CSV files.

In [22]:
validation_summary = pd.DataFrame(
    {
        "validation_layer": [
            "clean_matches",
            "standings",
            "team_season_stats",
        ],
        "passed": [
            clean_matches_checks_passed,
            standings_checks_passed,
            team_season_stats_checks_passed,
        ],
    }
)

display(validation_summary)

assert validation_summary["passed"].all()

,validation_layer,passed
0,clean_matches,True
1,standings,True
2,team_season_stats,True


In [23]:
dataset_summary = pd.DataFrame(
    [
        {
            "dataset": "all_matches",
            "rows": clean_matches.shape[0],
            "columns": clean_matches.shape[1],
            "output_path": f"data/processed/{all_matches_path.name}",
        },
        {
            "dataset": "participations",
            "rows": participations.shape[0],
            "columns": participations.shape[1],
            "output_path": f"data/processed/{participations_path.name}",
        },
        {
            "dataset": "team_participation_counts",
            "rows": team_participation_counts.shape[0],
            "columns": team_participation_counts.shape[1],
            "output_path": f"data/processed/{team_participation_counts_path.name}",
        },
        {
            "dataset": "standings",
            "rows": standings.shape[0],
            "columns": standings.shape[1],
            "output_path": f"data/processed/{standings_path.name}",
        },
        {
            "dataset": "team_season_stats",
            "rows": team_season_stats.shape[0],
            "columns": team_season_stats.shape[1],
            "output_path": f"data/processed/{team_season_stats_path.name}",
        },
    ]
)

display(dataset_summary)

,dataset,rows,columns,output_path
0,all_matches,3800,12,data/processed/all_matches.csv
1,participations,200,2,data/processed/participations.csv
2,team_participation_counts,33,3,data/processed/team_participation_counts.csv
3,standings,200,12,data/processed/standings.csv
4,team_season_stats,200,42,data/processed/team_season_stats.csv
